In [ ]:
# ============================================================================
# CELL 1: Install Packages
# ============================================================================
!pip install torch transformers==4.47.1 accelerate==0.26.1 scikit-learn numpy pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.9/270.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation:

In [ ]:
# ============================================================================
# CELL 2: Imports
# ============================================================================
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random
import re
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
# ============================================================================
# CELL 3: Load Data
# ============================================================================
train_path = "train_data (1).json"
test_path = "test_data_subtask_1 (1).json"

with open(train_path, 'r') as f:
    train_data = json.load(f)

with open(test_path, 'r') as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nValidity distribution:\n{train_df['validity'].value_counts()}")
print(f"\nPlausibility distribution:\n{train_df['plausibility'].value_counts()}")

# Analyze 4 conditions
conditions = train_df.groupby(['plausibility', 'validity']).size()
print(f"\n4-Condition Distribution (Original):")
print(conditions)

Training samples: 960
Test samples: 191

Validity distribution:
validity
False    480
True     480
Name: count, dtype: int64

Plausibility distribution:
plausibility
False    486
True     474
Name: count, dtype: int64

4-Condition Distribution (Original):
plausibility  validity
False         False       246
              True        240
True          False       234
              True        240
dtype: int64


In [ ]:
# ============================================================================
# CELL 4: PROPER Train/Val Split BEFORE Augmentation
# ============================================================================
print("\n" + "="*80)
print("Step 1: Split ORIGINAL Data First (Prevent Data Leakage)")
print("="*80)

# Create condition column for stratification
train_df['condition'] = (
    train_df['plausibility'].astype(str) + '_' +
    train_df['validity'].astype(str)
)

# Split ORIGINAL data first
train_indices, val_indices = train_test_split(
    range(len(train_df)),
    test_size=0.15,
    random_state=42,
    stratify=train_df['condition'].values
)

train_df_original = train_df.iloc[train_indices].copy()
val_df_original = train_df.iloc[val_indices].copy()

print(f"\nOriginal data split:")
print(f"  Training: {len(train_df_original)}")
print(f"  Validation: {len(val_df_original)}")
print(f"\n CRITICAL: Augmentation will ONLY be applied to training data")
print(f"   Validation stays ORIGINAL to test true generalization")


Step 1: Split ORIGINAL Data First (Prevent Data Leakage)

Original data split:
  Training: 816
  Validation: 144

 CRITICAL: Augmentation will ONLY be applied to training data
   Validation stays ORIGINAL to test true generalization


In [ ]:
# ============================================================================
# CELL 7: Dataset Class
# ============================================================================
class SyllogismDataset(Dataset):
    def __init__(self, texts, counterfactuals, validity_labels, plausibility_labels, tokenizer, max_length=512):
        self.texts = texts
        self.counterfactuals = counterfactuals
        self.validity_labels = validity_labels
        self.plausibility_labels = plausibility_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded_orig = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        encoded_cf = self.tokenizer(
            self.counterfactuals[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids_orig': encoded_orig['input_ids'].squeeze(0),
            'attention_mask_orig': encoded_orig['attention_mask'].squeeze(0),
            'input_ids_cf': encoded_cf['input_ids'].squeeze(0),
            'attention_mask_cf': encoded_cf['attention_mask'].squeeze(0),
            'validity_label': torch.tensor(self.validity_labels[idx], dtype=torch.long),
            'plausibility_label': torch.tensor(self.plausibility_labels[idx], dtype=torch.long)
        }

In [ ]:
# ============================================================================
# CELL 8: Focal Loss
# ============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
     inputs = inputs.float()  # ensure float32
     ce_loss = F.cross_entropy(inputs, targets, reduction='none')
     ce_loss = torch.clamp(ce_loss, max=100.0)  # prevent exp overflow
     p_t = torch.exp(-ce_loss)
     focal_loss = (1 - p_t) ** self.gamma * ce_loss

     if self.alpha is not None:
        alpha_t = self.alpha[targets]
        focal_loss = alpha_t * focal_loss

     if self.reduction == 'mean':
        return focal_loss.mean()
     elif self.reduction == 'sum':
        return focal_loss.sum()
     else:
        return focal_loss

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✓ Tokenizer loaded: {MODEL_NAME}")

✓ Tokenizer loaded: microsoft/deberta-v3-base


In [ ]:
# ============================================================================
# VANILLA DEBERTA BASELINE
# No augmentation, no adversarial, no focal loss, no oversampling, no multi-layer
# Just DeBERTa-v3-base + single [CLS] + CrossEntropy on original training data
# ============================================================================
import json
import gc
import numpy as np
from sklearn.metrics import accuracy_score

print("=" * 60)
print("EXPERIMENT: Vanilla DeBERTa Baseline")
print("=" * 60)
print(f"  Training data: {len(train_df_original)} samples (original, NO augmentation)")
print(f"  Validation data: {len(val_df_original)} samples (original)")
print(f"  Multi-layer: NO (single final [CLS])")
print(f"  Adversarial: NO")
print(f"  Focal loss: NO (standard CrossEntropy)")
print(f"  Oversampling: NO")

EXPERIMENT: Vanilla DeBERTa Baseline
  Training data: 816 samples (original, NO augmentation)
  Validation data: 144 samples (original)
  Multi-layer: NO (single final [CLS])
  Adversarial: NO
  Focal loss: NO (standard CrossEntropy)
  Oversampling: NO


In [ ]:
# ============================================================================
# Vanilla DeBERTa: single [CLS] layer, no adversarial head
# ============================================================================
class VanillaDeBERTa(nn.Module):
    """Bare DeBERTa-v3-base classifier. Single [CLS], no tricks."""

    def __init__(self, model_name, dropout=0.2):
        super(VanillaDeBERTa, self).__init__()
        self.encoder = AutoModel.from_pretrained(
            model_name,
            output_hidden_states=True,
            low_cpu_mem_usage=False
        )
        self.encoder = self.encoder.float()
        hidden_size = self.encoder.config.hidden_size  # 768

        # Simple classifier: 768 -> 256 -> 2
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use only final layer [CLS]
        cls_output = outputs.last_hidden_state[:, 0, :].float()
        logits = self.classifier(cls_output)
        return logits

print("✓ VanillaDeBERTa model defined")

✓ VanillaDeBERTa model defined


In [ ]:
# ============================================================================
# Prepare data — original training only, no augmentation, no oversampling
# ============================================================================
baseline_train = train_df_original.copy()

# Add counterfactual column (required by dataset class)
if 'counterfactual' not in baseline_train.columns:
    baseline_train['counterfactual'] = baseline_train['syllogism']

baseline_val = val_df_original.copy()
if 'counterfactual' not in baseline_val.columns:
    baseline_val['counterfactual'] = baseline_val['syllogism']

# Prepare arrays
bl_train_texts = baseline_train['syllogism'].tolist()
bl_train_cf = baseline_train['counterfactual'].tolist()
bl_train_validity = baseline_train['validity'].astype(int).values
bl_train_plausibility = baseline_train['plausibility'].astype(int).values

bl_val_texts = baseline_val['syllogism'].tolist()
bl_val_cf = baseline_val['counterfactual'].tolist()
bl_val_validity = baseline_val['validity'].astype(int).values
bl_val_plausibility = baseline_val['plausibility'].astype(int).values

# Datasets
bl_train_dataset = SyllogismDataset(
    bl_train_texts, bl_train_cf, bl_train_validity,
    bl_train_plausibility, tokenizer
)
bl_val_dataset = SyllogismDataset(
    bl_val_texts, bl_val_cf, bl_val_validity,
    bl_val_plausibility, tokenizer
)

# Dataloaders
BATCH_SIZE = 4
bl_train_loader = DataLoader(bl_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
bl_val_loader = DataLoader(bl_val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(baseline_train)}")
print(f"Validation samples: {len(baseline_val)}")
print(f"Train batches: {len(bl_train_loader)}")
print(f"Val batches: {len(bl_val_loader)}")

Training samples: 816
Validation samples: 144
Train batches: 204
Val batches: 36


In [ ]:
# ============================================================================
# Initialize vanilla model
# ============================================================================
EPOCHS = 4
LR = 2e-5
ACCUMULATION_STEPS = 4

bl_model = VanillaDeBERTa(MODEL_NAME, dropout=0.2)
bl_model = bl_model.to(device)

bl_criterion = nn.CrossEntropyLoss()

bl_optimizer = torch.optim.AdamW(bl_model.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(bl_train_loader) // ACCUMULATION_STEPS) * EPOCHS
bl_scheduler = get_linear_schedule_with_warmup(
    bl_optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Model parameters: {sum(p.numel() for p in bl_model.parameters()):,}")
print(f"  (vs MLA-CI: 187,769,860)")
print(f"Epochs: {EPOCHS}")
print(f"Effective batch size: {BATCH_SIZE * ACCUMULATION_STEPS}")

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Model parameters: 184,029,442
  (vs MLA-CI: 187,769,860)
Epochs: 4
Effective batch size: 16


In [ ]:
# ============================================================================
# Train vanilla baseline
# ============================================================================
print("\n" + "#" * 80)
print("# TRAINING: Vanilla DeBERTa Baseline")
print("#" * 80)

best_score = 0
best_results = {}
all_epoch_results = []

for epoch in range(EPOCHS):
    # --- TRAIN ---
    bl_model.train()
    total_loss = 0
    correct = 0
    total = 0
    bl_optimizer.zero_grad()

    for batch_idx, batch in enumerate(tqdm(bl_train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        input_ids = batch['input_ids_orig'].to(device)
        attention_mask = batch['attention_mask_orig'].to(device)
        val_labels = batch['validity_label'].to(device)

        logits = bl_model(input_ids, attention_mask)
        loss = bl_criterion(logits, val_labels) / ACCUMULATION_STEPS

        loss.backward()

        if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(bl_train_loader):
            torch.nn.utils.clip_grad_norm_(bl_model.parameters(), 1.0)
            bl_optimizer.step()
            bl_scheduler.step()
            bl_optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS
        preds = torch.argmax(logits, dim=1)
        correct += (preds == val_labels).sum().item()
        total += val_labels.size(0)

    train_acc = correct / total

    # --- EVALUATE ---
    bl_model.eval()
    all_preds = []
    all_val = []
    all_plaus = []
    idx = 0

    with torch.no_grad():
        for batch in bl_val_loader:
            input_ids = batch['input_ids_orig'].to(device)
            attention_mask = batch['attention_mask_orig'].to(device)
            val_labels = batch['validity_label'].to(device)
            bs = val_labels.size(0)

            logits = bl_model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_val.extend(val_labels.cpu().numpy())
            for i in range(bs):
                all_plaus.append(bl_val_plausibility[idx])
                idx += 1

    all_preds = np.array(all_preds)
    all_val = np.array(all_val)
    all_plaus = np.array(all_plaus)

    # Overall accuracy
    val_acc = accuracy_score(all_val, all_preds)

    # Per-condition accuracy
    conditions = {}
    for plaus in [0, 1]:
        for valid in [0, 1]:
            mask = (all_plaus == plaus) & (all_val == valid)
            if mask.sum() > 0:
                acc = accuracy_score(all_val[mask], all_preds[mask])
                plaus_name = "Plausible" if plaus == 1 else "Implausible"
                valid_name = "Valid" if valid == 1 else "Invalid"
                conditions[f"{plaus_name}_{valid_name}"] = acc

    # Content effect
    intra_plaus = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Plausible_Invalid', 0))
    intra_implaus = abs(conditions.get('Implausible_Valid', 0) - conditions.get('Implausible_Invalid', 0))
    intra_effect = (intra_plaus + intra_implaus) / 2

    cross_valid = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Implausible_Valid', 0))
    cross_invalid = abs(conditions.get('Plausible_Invalid', 0) - conditions.get('Implausible_Invalid', 0))
    cross_effect = (cross_valid + cross_invalid) / 2

    val_ce = (intra_effect + cross_effect) / 2

    # Combined score
    combined = val_acc * 100 / (1 + np.log(1 + val_ce * 100))

    print(f"\n  Epoch {epoch+1}: Train Acc={train_acc*100:.2f}% | Val Acc={val_acc*100:.2f}% | CE={val_ce*100:.2f}% | Score={combined:.2f}")
    print(f"    PV={conditions.get('Plausible_Valid',0)*100:.1f}  PI={conditions.get('Plausible_Invalid',0)*100:.1f}  "
          f"IV={conditions.get('Implausible_Valid',0)*100:.1f}  II={conditions.get('Implausible_Invalid',0)*100:.1f}")

    epoch_result = {
        'epoch': epoch + 1,
        'train_acc': round(train_acc * 100, 2),
        'val_acc': round(val_acc * 100, 2),
        'val_ce': round(val_ce * 100, 2),
        'combined_score': round(combined, 2),
        'conditions': {k: round(v * 100, 2) for k, v in conditions.items()}
    }
    all_epoch_results.append(epoch_result)

    if combined > best_score:
        best_score = combined
        best_results = {
            'experiment': 'Vanilla DeBERTa (baseline)',
            'best_epoch': epoch + 1,
            'val_acc': round(val_acc * 100, 2),
            'val_ce': round(val_ce * 100, 2),
            'combined_score': round(combined, 2),
            'conditions': {k: round(v * 100, 2) for k, v in conditions.items()},
        }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

best_results['all_epochs'] = all_epoch_results

print(f"\n{'=' * 60}")
print(f"★ BEST: Epoch {best_results['best_epoch']} | Acc={best_results['val_acc']}% | CE={best_results['val_ce']}% | Score={best_results['combined_score']}")
print(f"{'=' * 60}")


################################################################################
# TRAINING: Vanilla DeBERTa Baseline
################################################################################


Epoch 1/4: 100%|██████████| 204/204 [01:57<00:00,  1.74it/s]



  Epoch 1: Train Acc=54.04% | Val Acc=68.06% | CE=13.61% | Score=18.48
    PV=52.8  PI=80.0  IV=61.1  II=78.4


Epoch 2/4: 100%|██████████| 204/204 [02:00<00:00,  1.70it/s]



  Epoch 2: Train Acc=77.45% | Val Acc=78.47% | CE=5.95% | Score=26.70
    PV=83.3  PI=71.4  IV=83.3  II=75.7


Epoch 3/4: 100%|██████████| 204/204 [02:00<00:00,  1.70it/s]



  Epoch 3: Train Acc=85.91% | Val Acc=77.08% | CE=20.23% | Score=19.01
    PV=88.9  PI=65.7  IV=97.2  II=56.8


Epoch 4/4: 100%|██████████| 204/204 [02:00<00:00,  1.69it/s]



  Epoch 4: Train Acc=90.93% | Val Acc=81.94% | CE=5.74% | Score=28.17
    PV=75.0  PI=85.7  IV=80.6  II=86.5

★ BEST: Epoch 4 | Acc=81.94% | CE=5.74% | Score=28.17


In [ ]:
# ============================================================================
# Save baseline results
# ============================================================================
with open('vanilla_baseline_results.json', 'w') as f:
    json.dump(best_results, f, indent=2)
print(" Saved to vanilla_baseline_results.json")

# Print comparison with existing results
print(f"\n{'=' * 70}")
print("COMPARISON WITH EXISTING RESULTS")
print(f"{'=' * 70}")
print(f"{'Configuration':<35} {'Acc (%)':<10} {'CE (%)':<10} {'Score':<10}")
print("-" * 65)
print(f"{'Vanilla DeBERTa (baseline)':<35} {best_results['val_acc']:<10} {best_results['val_ce']:<10} {best_results['combined_score']:<10}  ← NEW")
print(f"{'Full MLA-CI (submitted)':<35} {'83.33':<10} {'8.33':<10} {'25.77':<10}")
print(f"{'− Adversarial (best config)':<35} {'93.06':<10} {'4.20':<10} {'35.12':<10}")
print(f"{'− Augmentation':<35} {'72.92':<10} {'14.44':<10} {'19.51':<10}")
print(f"{'− Multi-layer':<35} {'71.53':<10} {'20.07':<10} {'17.67':<10}")
print("-" * 65)

# Cleanup
del bl_model, bl_optimizer, bl_scheduler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\n✓ GPU memory cleared")

 Saved to vanilla_baseline_results.json

COMPARISON WITH EXISTING RESULTS
Configuration                       Acc (%)    CE (%)     Score     
-----------------------------------------------------------------
Vanilla DeBERTa (baseline)          81.94      5.74       28.17       ← NEW
Full MLA-CI (submitted)             83.33      8.33       25.77     
− Adversarial (best config)         93.06      4.20       35.12     
− Augmentation                      72.92      14.44      19.51     
− Multi-layer                       71.53      20.07      17.67     
-----------------------------------------------------------------

✓ GPU memory cleared
